# Building the Backend for a Full-Stack LangGraph App

## Step-by-Step Guide: From Notebook Agent to FastAPI Backend

In the notebook `028-1st-top-use-case-langgraph-2026.ipynb`, we built a **LangGraph customer service agent** for BigCompany Inc. that runs inside a Jupyter notebook.

That's great for learning and testing, but in the real world, an AI agent needs to be **accessible from the web**. A customer should be able to chat with the agent through a website, not through a Jupyter notebook.

To make that happen, we need to wrap our LangGraph agent inside a **backend server** — just like the To Do app in folder `050-todoapp-LC-backend` uses FastAPI to expose its features as an API.

**In this notebook, we will create the backend step by step.**

## What is a Backend? (Quick Refresher)

Think of it this way:

- **The LangGraph agent** is the "brain" — it knows how to answer customer questions, look up orders, and process returns.
- **The backend (FastAPI)** is the "reception desk" — it receives messages from the website, passes them to the brain, and sends the answer back.
- **The frontend** (which we'll build later) is the "chat window" on the website that the customer sees.

```
┌──────────────┐     HTTP Request      ┌──────────────────┐     invoke()     ┌──────────────────┐
│              │  ─────────────────►   │                  │  ─────────────►  │                  │
│   Frontend   │   POST /chat          │  FastAPI Backend │                  │  LangGraph Agent │
│  (Website)   │   {message, thread}   │  (Reception)     │                  │  (Brain)         │
│              │  ◄─────────────────   │                  │  ◄─────────────  │                  │
└──────────────┘     HTTP Response     └──────────────────┘     response     └──────────────────┘
```

The backend is just a **bridge** between the frontend and the agent.

## How Does This Compare to the basic To Do App Backend we built in Bootcamp #1?

In the To Do app (`050-todoapp-LC-backend`), the backend:
- Uses **FastAPI** as the web framework.
- Has **API endpoints** (like `POST /todos`, `GET /todos`) for creating and reading to-do items.
- Uses **LangChain** to add AI features (summarize text, write poems).
- Connects to a **PostgreSQL database** with SQLAlchemy.

Our new backend will be **simpler**:
- Uses **FastAPI** as the web framework (same).
- Has **one main API endpoint**: `POST /chat` to send messages to the agent.
- Uses **LangGraph** instead of LangChain (the agent from notebook 028).
- **No database needed** — LangGraph's built-in memory handles conversation history.

| Feature | To Do App Backend (050) | Our LangGraph Backend |
|---------|------------------------|----------------------|
| Web framework | FastAPI | FastAPI |
| AI framework | LangChain (simple chain) | LangGraph (agent with tools) |
| Database | PostgreSQL + SQLAlchemy | None (in-memory state) |
| Main endpoint | CRUD endpoints (`/todos`) | Chat endpoint (`/chat`) |
| CORS middleware | Yes | Yes |

## Project Structure

Here are the files we will create:

```
backend/
├── .env                  # Secret keys (OpenAI API key)
├── requirements.txt      # Python packages we need
├── main.py               # FastAPI app (the "reception desk")
├── agent.py              # LangGraph agent (the "brain")
├── schemas.py            # Data shapes for requests and responses
└── routers/
    └── chat.py           # The /chat API endpoint
```

Compare this to the To Do app (`050-todoapp-LC-backend`):

```
050-todoapp-LC-backend/
├── .env                  # Secret keys
├── pyproject.toml        # Python packages
├── main.py               # FastAPI app           ← same role
├── config.py             # Settings
├── database.py           # Database connection    ← we don't need this
├── models.py             # Database models        ← we don't need this
├── schemas.py            # Data shapes            ← same role
├── crud.py               # Database operations    ← we don't need this
└── routers/
    └── todos.py          # API endpoints          ← same role (ours: chat.py)
```

See the pattern? Our backend is **simpler** because we don't need a database. The LangGraph agent handles its own state.

Let's build it file by file!

## Step 1: Create the Project Folder

First, let's create the `backend/` folder and the `routers/` subfolder inside it.

In [ ]:
import os

# Create the backend folder and routers subfolder
os.makedirs("backend/routers", exist_ok=True)

print("Folders created!")
print("  backend/")
print("  backend/routers/")

## Step 2: Create the `.env` File (Secret Keys)

The `.env` file stores our **OpenAI API key**. This key is needed for the LLM (the agent's brain) to work.

**Important**: Never share your API key or commit it to Git!

In the To Do app, the `.env` file had database credentials AND an OpenAI key. Ours only needs the OpenAI key because we don't use a database.

In [ ]:
%%writefile backend/.env
OPENAI_API_KEY=your-openai-api-key-here

**After running this cell**, open the file `backend/.env` and replace `your-openai-api-key-here` with your actual OpenAI API key.

## Step 3: Create `requirements.txt` (Dependencies)

This file lists all the Python packages our backend needs.

| Package | What It Does |
|---------|--------------|
| `fastapi` | The web framework (handles HTTP requests) |
| `uvicorn[standard]` | The server that runs our FastAPI app |
| `langgraph` | The framework for our AI agent's workflow |
| `langchain-openai` | Connects to OpenAI's GPT models |
| `langchain-core` | Core utilities (message types, tools) |
| `python-dotenv` | Loads the `.env` file |
| `pydantic` | Data validation for requests/responses |

In the To Do app, these were listed in `pyproject.toml` (managed by Poetry). For simplicity, we'll use a plain `requirements.txt` (managed by pip).

In [ ]:
%%writefile backend/requirements.txt
fastapi>=0.115.0
uvicorn[standard]>=0.34.0
langgraph>=0.2.0
langchain-openai>=0.3.0
langchain-core>=0.3.0
python-dotenv>=1.0.0
pydantic>=2.0.0

To install these packages, run this command in your terminal:

```bash
pip install -r backend/requirements.txt
```

## Step 4: Create `agent.py` (The LangGraph Agent)

This is the **heart of our backend**. It contains the same LangGraph agent we built in notebook 028, but organized in a single Python file that can be imported by the API.

Here's what this file does:

1. **Defines the fake database** (orders, return policy) — same as notebook 028.
2. **Defines the tools** (check_order_status, lookup_return_policy, process_return_request) — same as notebook 028.
3. **Sets up the LLM** with tools bound — same as notebook 028.
4. **Builds the LangGraph graph** (assistant node, tools node, conditional edges) — same as notebook 028.
5. **Exports the `graph` object** so the API can use it.

The only difference from notebook 028: all the code is in **one file** instead of spread across multiple notebook cells.

In [ ]:
%%writefile backend/agent.py
# ============================================================
# agent.py — The LangGraph Customer Service Agent
#
# This file contains the SAME agent from notebook 028,
# packaged as a module that the FastAPI backend can import.
# ============================================================

import os
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver

# Load environment variables (the OpenAI API key)
load_dotenv()


# ============================================================
# 1. THE "DATABASE" (simulated with dictionaries)
# ============================================================
# In a real app, this data would come from a real database.
# For this demo, we use simple Python dictionaries.

ORDERS_DB = {
    "ORD-1001": {
        "customer": "Alice Johnson",
        "product": "Laptop Pro 15",
        "price": 1299.99,
        "status": "delivered",
        "delivery_date": "2026-02-05",
    },
    "ORD-1002": {
        "customer": "Bob Smith",
        "product": "Wireless Headphones X3",
        "price": 199.99,
        "status": "shipped",
        "estimated_delivery": "2026-02-14",
    },
    "ORD-1003": {
        "customer": "Carol Davis",
        "product": "Smartphone Ultra 12",
        "price": 899.99,
        "status": "processing",
        "estimated_delivery": "2026-02-18",
    },
}

RETURN_POLICY = {
    "standard_return_window": "30 days from delivery",
    "refund_processing_time": "5-7 business days",
    "conditions": [
        "Item must be in original packaging",
        "Item must not show signs of damage caused by the customer",
        "Proof of purchase (order ID) is required",
    ],
    "non_returnable": ["Earbuds (hygiene reasons)", "Software licenses"],
}


# ============================================================
# 2. THE TOOLS (functions the agent can call)
# ============================================================
# These are the same 3 tools from notebook 028.
# The @tool decorator makes them usable by the LLM.

@tool
def check_order_status(order_id: str) -> str:
    """Check the current status of a customer order.

    Use this tool when a customer asks about their order status,
    delivery date, or order details.

    Args:
        order_id: The order ID (e.g., 'ORD-1001').
    """
    order = ORDERS_DB.get(order_id)

    if not order:
        return f"Order '{order_id}' not found. Please check the order ID and try again."

    info = f"Order {order_id}:\n"
    info += f"  Customer: {order['customer']}\n"
    info += f"  Product: {order['product']}\n"
    info += f"  Price: ${order['price']}\n"
    info += f"  Status: {order['status']}\n"

    if order["status"] == "delivered":
        info += f"  Delivered on: {order['delivery_date']}"
    else:
        info += f"  Estimated delivery: {order['estimated_delivery']}"

    return info


@tool
def lookup_return_policy() -> str:
    """Look up BigCompany Inc.'s return and refund policy.

    Use this tool when a customer asks about returns, refunds,
    or the return policy.
    """
    policy = "BigCompany Inc. Return Policy:\n"
    policy += f"  Return window: {RETURN_POLICY['standard_return_window']}\n"
    policy += f"  Refund processing: {RETURN_POLICY['refund_processing_time']}\n"
    policy += "  Conditions:\n"
    for condition in RETURN_POLICY["conditions"]:
        policy += f"    - {condition}\n"
    policy += "  Non-returnable items:\n"
    for item in RETURN_POLICY["non_returnable"]:
        policy += f"    - {item}\n"
    return policy


@tool
def process_return_request(order_id: str, reason: str) -> str:
    """Process a return request for a customer order.

    Use this tool when a customer confirms they want to return a product.
    Only use this AFTER checking the order status and confirming
    the customer wants to proceed.

    Args:
        order_id: The order ID to process the return for.
        reason: The customer's reason for returning the product.
    """
    order = ORDERS_DB.get(order_id)

    if not order:
        return f"Cannot process return: Order '{order_id}' not found."

    if order["status"] != "delivered":
        return (
            f"Cannot process return: Order {order_id} has status '{order['status']}'. "
            f"Returns can only be processed for delivered orders."
        )

    return (
        f"Return request APPROVED for order {order_id} ({order['product']}).\n"
        f"  Reason: {reason}\n"
        f"  Refund amount: ${order['price']}\n"
        f"  Refund will be processed within {RETURN_POLICY['refund_processing_time']}.\n"
        f"  A return shipping label has been sent to the customer's email."
    )


# Collect all tools in a list
tools = [check_order_status, lookup_return_policy, process_return_request]


# ============================================================
# 3. THE LLM (the agent's brain)
# ============================================================

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_with_tools = llm.bind_tools(tools)

SYSTEM_PROMPT = """You are a friendly and helpful customer service agent for BigCompany Inc., \
an online electronics store.

Your responsibilities:
- Help customers check their order status.
- Answer questions about our return and refund policy.
- Process return requests when customers want to return a product.
- Be polite, empathetic, and professional at all times.

Important guidelines:
- Always greet the customer warmly.
- When a customer asks about an order, use the check_order_status tool.
- When a customer asks about returns or refunds, use the lookup_return_policy tool.
- Before processing a return, always confirm with the customer first.
- If you cannot help with something, let the customer know you will \
connect them with a human agent.
- Keep responses concise but helpful.
"""


# ============================================================
# 4. THE GRAPH (the agent's workflow)
# ============================================================

# Node 1: The Assistant (calls the LLM)
def assistant(state: MessagesState):
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


# Node 2: The Tool Executor
tool_node = ToolNode(tools)

# Build the graph
builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant)
builder.add_node("tools", tool_node)
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")

# Compile with memory (so the agent remembers conversations)
memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)

print("BigCompany Inc. LangGraph agent loaded successfully!")

### What just happened?

We created `backend/agent.py` — a single file that contains our entire LangGraph agent. This is the same agent from notebook 028, just packaged as a Python module.

The key thing is the **last line**: `graph = builder.compile(checkpointer=memory)`. This creates the `graph` object that our API endpoint will import and use.

```python
# Later, in our API endpoint, we'll do:
from agent import graph

result = graph.invoke({"messages": [...]}, config)
```

## Step 5: Create `schemas.py` (Request/Response Shapes)

**Schemas** define the shape of the data going in and out of our API. They are like a **contract**: they tell the frontend exactly what to send and what to expect back.

In the To Do app, schemas defined `ToDoRequest` (name + completed) and `ToDoResponse` (name + completed + id).

For our chat app, we need:

| Schema | Fields | Used For |
|--------|--------|---------|
| `ChatRequest` | `message` + `thread_id` | What the frontend **sends** to the API |
| `ChatResponse` | `response` + `thread_id` | What the API **sends back** to the frontend |

The `thread_id` is important — it tells the agent **which conversation** this message belongs to. Same thread_id = same conversation (the agent remembers previous messages).

In [ ]:
%%writefile backend/schemas.py
# ============================================================
# schemas.py — Data shapes for the Chat API
#
# Pydantic models define what data goes in and out of the API.
# Think of them as a "contract" between the frontend and backend.
# ============================================================

from pydantic import BaseModel


class ChatRequest(BaseModel):
    """What the frontend sends to the backend."""
    message: str       # The customer's message (e.g., "What's my order status?")
    thread_id: str     # Conversation ID (same ID = same conversation)


class ChatResponse(BaseModel):
    """What the backend sends back to the frontend."""
    response: str      # The agent's reply
    thread_id: str     # Same conversation ID (so the frontend knows which conversation)

### Compare with the To Do app schemas

To Do app (`050-todoapp-LC-backend/schemas.py`):
```python
class ToDoRequest(BaseModel):
    name: str
    completed: bool

class ToDoResponse(BaseModel):
    name: str
    completed: bool
    id: int
```

Our chat app (`backend/schemas.py`):
```python
class ChatRequest(BaseModel):
    message: str
    thread_id: str

class ChatResponse(BaseModel):
    response: str
    thread_id: str
```

Same pattern, different fields. Simple!

## Step 6: Create `routers/chat.py` (The Chat API Endpoint)

This is where the magic happens. This file creates the **`POST /chat`** endpoint — the door through which the frontend talks to the agent.

In the To Do app, `routers/todos.py` had multiple endpoints (POST, GET, PUT, DELETE for to-do items). Our chat app only needs **one endpoint**: `POST /chat`.

Here's what the endpoint does:

1. **Receives** the customer's message and thread_id from the frontend.
2. **Passes** the message to the LangGraph agent.
3. **Waits** for the agent to think, use tools, and generate a response.
4. **Returns** the agent's response to the frontend.

In [ ]:
%%writefile backend/routers/chat.py
# ============================================================
# routers/chat.py — The Chat API Endpoint
#
# This is the "door" between the frontend and the LangGraph agent.
# It receives messages from the frontend, passes them to the agent,
# and sends the agent's response back.
# ============================================================

from fastapi import APIRouter
from langchain_core.messages import HumanMessage

from schemas import ChatRequest, ChatResponse
from agent import graph

router = APIRouter()


@router.post("/chat", response_model=ChatResponse)
async def chat(request: ChatRequest):
    """
    Send a message to the BigCompany Inc. customer service agent.

    - The `message` is what the customer typed.
    - The `thread_id` identifies the conversation.
      Same thread_id = same conversation (the agent remembers context).
    """

    # The config tells LangGraph which conversation thread to use.
    # This is how the agent "remembers" previous messages.
    config = {"configurable": {"thread_id": request.thread_id}}

    # Send the customer's message to the LangGraph agent.
    # The agent will:
    #   1. Read the message
    #   2. Decide if it needs to use a tool
    #   3. If yes: call the tool, get results, think again
    #   4. Generate a final response
    result = graph.invoke(
        {"messages": [HumanMessage(content=request.message)]},
        config,
    )

    # The result contains ALL messages in the conversation.
    # The last message is the agent's final response.
    agent_response = result["messages"][-1].content

    return ChatResponse(
        response=agent_response,
        thread_id=request.thread_id,
    )

### Compare with the To Do app router

To Do app (`050-todoapp-LC-backend/routers/todos.py`) — had many endpoints:
```python
@router.post("")           # Create a to-do
@router.get("")            # List all to-dos
@router.get("/{id}")       # Get one to-do
@router.put("/{id}")       # Update a to-do
@router.delete("/{id}")    # Delete a to-do
@router.post("/summarize-text")  # LangChain: summarize
@router.post("/write-poem/{id}") # LangChain: write poem
```

Our chat app (`backend/routers/chat.py`) — just one:
```python
@router.post("/chat")      # Chat with the agent
```

One endpoint is all we need. The LangGraph agent handles all the complexity internally (deciding which tool to use, managing conversation state, etc.).

## Step 7: Create `main.py` (The FastAPI App)

This is the **entry point** of our backend. It creates the FastAPI app, adds CORS middleware, and includes the chat router.

**What is CORS?** When a frontend (running on `localhost:3000`) tries to call a backend (running on `localhost:8000`), the browser blocks it for security reasons. CORS middleware tells the browser: *"It's OK, I trust this frontend."*

The To Do app's `main.py` had the same CORS setup. We're using the exact same pattern.

In [ ]:
%%writefile backend/main.py
# ============================================================
# main.py — The FastAPI Application
#
# This is the entry point of the backend.
# It creates the FastAPI app, adds CORS, and includes the router.
# ============================================================

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

from routers import chat

# Create the FastAPI app
app = FastAPI(
    title="BigCompany Inc. Customer Service API",
    description="A LangGraph-powered customer service agent backend.",
)

# CORS middleware — allows the frontend to call our API.
# Without this, the browser would block requests from the frontend.
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000"],  # Frontend URL
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Include the chat router (this adds the /chat endpoint)
app.include_router(chat.router)


# A simple health check endpoint
@app.get("/")
def read_root():
    return {"message": "BigCompany Inc. Customer Service API is running!"}

### Compare with the To Do app `main.py`

The structure is almost identical:

| Part | To Do App (050) | Our Chat App |
|------|----------------|--------------|
| Create app | `app = FastAPI()` | `app = FastAPI(title=..., description=...)` |
| CORS | `allow_origins=["http://localhost:3000"]` | Same |
| Include router | `app.include_router(todos.router)` | `app.include_router(chat.router)` |
| Root endpoint | `return "Hello World"` | `return {"message": "...API is running!"}` |

The To Do app also had settings via `config.py` and a database connection. We don't need those, so our `main.py` is simpler.

## Step 8: Create the `routers/__init__.py` file

This small file tells Python that the `routers/` folder is a package that can be imported. Without it, `from routers import chat` would not work.

In [ ]:
%%writefile backend/routers/__init__.py
# This file makes the routers/ folder a Python package.

#### Simple Explanation

Think of the `__init__.py` file like a **door sign** on a room.

You have a folder called `routers/` with useful code inside. Python won't recognize it as a proper "package" (a group of related code files) unless you put an `__init__.py` file inside it — even if that file is completely empty.

**Without it:** Python says *"I don't know what `routers` is"* when you try to `from routers import chat`.

**With it:** Python says *"Ah, `routers` is a package, let me look inside it."*

In this case, the file just has a comment and nothing else — its mere **existence** is what matters, not its content.

**It's a Python convention that, while less strictly required in modern Python 3, is still considered a best practice, especially in project structures like FastAPI backends**.

## All Files Created! Let's Review

Let's verify that all our backend files were created correctly.

In [ ]:
import os

print("Backend project structure:")
print("=" * 40)
for root, dirs, files in os.walk("backend"):
    # Skip hidden folders like __pycache__
    dirs[:] = [d for d in dirs if not d.startswith(".") and d != "__pycache__"]
    level = root.replace("backend", "").count(os.sep)
    indent = "  " * level
    folder_name = os.path.basename(root)
    print(f"{indent}{folder_name}/")
    sub_indent = "  " * (level + 1)
    for file in sorted(files):
        if not file.startswith(".") or file == ".env":
            filepath = os.path.join(root, file)
            size = os.path.getsize(filepath)
            print(f"{sub_indent}{file} ({size} bytes)")

## Step 9: Run the Backend

Now let's start the server! We'll use **uvicorn**, which is the same server the To Do app uses.

### Option A: Run from the terminal

Open a terminal, navigate to the `backend/` folder, and run:

```bash
cd backend
uvicorn main:app --reload
```

- `main:app` means: in the file `main.py`, use the object called `app`.
- `--reload` means: automatically restart when you change code (useful during development).

The server will start on **http://localhost:8000**.

### Option B: Run from this notebook

You can also start the server from this notebook (it will run in the background). **Note**: Make sure you have added your OpenAI API key to `backend/.env` first!

In [ ]:
# Uncomment and run this cell to start the server from the notebook.
# Press the stop button (or restart the kernel) to stop it.

# import subprocess
# subprocess.Popen(["uvicorn", "main:app", "--reload"], cwd="backend")

## Step 10: Test the Backend

Once the server is running, we can test it! There are 3 ways to test:

### Way 1: Swagger UI (Easiest)

FastAPI automatically creates a beautiful interactive documentation page. Open your browser and go to:

**http://localhost:8000/docs**

You'll see the `/chat` endpoint. Click on it, click "Try it out", and send a test message:

```json
{
  "message": "Hi! What's the status of order ORD-1002?",
  "thread_id": "test-session-1"
}
```

### Way 2: Test from this notebook with Python

We can also test by sending HTTP requests from this notebook (make sure the server is running first!).

In [ ]:
import requests

API_URL = "http://localhost:8000"

# Test 1: Health check
print("Test 1: Health check")
print("=" * 40)
response = requests.get(f"{API_URL}/")
print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")
print()

In [ ]:
# Test 2: Simple greeting
print("Test 2: Simple greeting")
print("=" * 40)
response = requests.post(
    f"{API_URL}/chat",
    json={
        "message": "Hi! I need some help.",
        "thread_id": "test-session-1",
    },
)
print(f"Status: {response.status_code}")
data = response.json()
print(f"Agent: {data['response']}")
print(f"Thread: {data['thread_id']}")
print()

In [ ]:
# Test 3: Order status check (the agent will use the check_order_status tool)
print("Test 3: Order status check")
print("=" * 40)
response = requests.post(
    f"{API_URL}/chat",
    json={
        "message": "What's the status of order ORD-1002?",
        "thread_id": "test-session-1",
    },
)
data = response.json()
print(f"Agent: {data['response']}")
print()

In [ ]:
# Test 4: Follow-up question (agent should remember ORD-1002 from the same thread)
print("Test 4: Follow-up (tests conversation memory)")
print("=" * 40)
response = requests.post(
    f"{API_URL}/chat",
    json={
        "message": "When will it arrive?",
        "thread_id": "test-session-1",  # Same thread_id!
    },
)
data = response.json()
print(f"Agent: {data['response']}")
print()

In [ ]:
# Test 5: Return policy question
print("Test 5: Return policy question")
print("=" * 40)
response = requests.post(
    f"{API_URL}/chat",
    json={
        "message": "What is your return policy?",
        "thread_id": "test-session-2",  # Different customer, different thread
    },
)
data = response.json()
print(f"Agent: {data['response']}")
print()

### Way 3: Test with curl (from the terminal)

You can also test from the terminal using `curl`:

```bash
curl -X POST http://localhost:8000/chat \
  -H "Content-Type: application/json" \
  -d '{"message": "What is the status of order ORD-1001?", "thread_id": "test-1"}'
```

## Recap: What We Built

We created a **FastAPI backend** that wraps the LangGraph customer service agent from notebook 028. Here's a summary of each file and what it does:

| File | Role | Lines of Code |
|------|------|---|
| `.env` | Stores the OpenAI API key | 1 |
| `requirements.txt` | Lists the Python packages needed | 7 |
| `agent.py` | The LangGraph agent (tools, LLM, graph) — the brain | ~140 |
| `schemas.py` | Pydantic models for request/response shapes | ~12 |
| `main.py` | FastAPI app with CORS middleware | ~20 |
| `routers/chat.py` | The POST /chat API endpoint | ~25 |
| `routers/__init__.py` | Makes `routers/` a Python package | 1 |

### The Flow (How Everything Connects)

```
Frontend sends POST /chat
  with {message: "Check order ORD-1002", thread_id: "session-1"}
        │
        ▼
main.py receives the request
  └── Routes it to routers/chat.py
        │
        ▼
routers/chat.py
  └── Calls graph.invoke() from agent.py
        │
        ▼
agent.py (the LangGraph agent)
  ├── Assistant node: LLM reads the message
  ├── LLM decides to call check_order_status tool
  ├── Tools node: executes the tool, gets order data
  ├── Assistant node: LLM reads tool results
  └── LLM generates the final response
        │
        ▼
routers/chat.py
  └── Returns {response: "Your order is shipped...", thread_id: "session-1"}
        │
        ▼
Frontend displays the agent's response
```

## Key Takeaways

1. **The backend is just a bridge.** It receives HTTP requests from the frontend, passes them to the LangGraph agent, and returns the response. That's it.

2. **The agent is the same one from notebook 028.** We didn't change the agent at all — we just moved it into its own file (`agent.py`) so the API can import it.

3. **FastAPI makes it easy.** Creating an API endpoint with FastAPI is just a function with a decorator (`@router.post("/chat")`). Pydantic handles data validation automatically.

4. **CORS is essential.** Without it, the browser would block the frontend from calling our API. Copy the same CORS setup from the To Do app.

5. **Thread IDs enable conversation memory.** By passing a `thread_id` with each message, the agent knows which conversation the message belongs to and can remember context.

## What's Next?

Now that the backend is working, the next step is to build the **frontend** — a chat interface where customers can type messages and see the agent's responses.

The frontend will be similar to the To Do app frontend in `051-todoapp-LC-frontend`, but instead of a to-do list, it will show a **chat window** that sends messages to our `POST /chat` endpoint.